# CELL 00 — Binary reproduction target

Rebuilds the supplied S³ spectral-spatial domain-adaptive EEG implementation for **Left Hand MI vs Right Hand MI**. Only the task/output and fixed evaluation-subject list are changed; the original architecture and training hyperparameters are retained.

Runs 4, 8, 12 only: T1 → class 0 (Left Hand MI), T2 → class 1 (Right Hand MI). The `SimplifiedBiMamba` name is retained from the supplied code; its implementation is a bidirectional GRU, not a true Mamba SSM.

In [1]:
# CELL 01 — Imports
from pathlib import Path
import os, math, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, cohen_kappa_score,
                             confusion_matrix, classification_report, roc_curve, roc_auc_score, auc)
from sklearn.manifold import TSNE
warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

In [2]:
# CELL 02 — Original configuration + fixed folds
SEED = 42
DATA_DIR = "./eegmmidb"
TOTAL_SUBJECTS = 109
TEST_SUBJECTS = [4, 15, 23, 29, 31, 42, 55, 71, 82, 95]
NUM_TEST_FOLDS = len(TEST_SUBJECTS)
NUM_TRAIN_SUBJECTS = 99
RUNS = [4, 8, 12]
TMIN, TMAX, FS = 0.0, 4.0, 250.0
N_CHANNELS = 22
N_CLASSES = 2
CLASS_NAMES = ["Left Hand MI", "Right Hand MI"]
BATCH_SIZE = 64
NUM_EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
DOMAIN_WEIGHT = 1.0
SUPCON_WEIGHT = 0.5
SUPCON_TEMP = 0.07
GRAD_CLIP = 1.0
NUM_FILTERS = 10
SINC_KERNEL = 81
SPATIAL_DIM = 64
DOMAIN_CLASSES = TOTAL_SUBJECTS
RESULTS_DIR = Path("./results_s3_da_binary_original_reproduction")
FIG_DIR = RESULTS_DIR / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()
if torch.cuda.is_available(): DEVICE=torch.device("cuda")
elif torch.backends.mps.is_available(): DEVICE=torch.device("mps")
else: DEVICE=torch.device("cpu")
print("Device:", DEVICE)
print("Fixed test subjects:", [f"S{s:03d}" for s in TEST_SUBJECTS])

Device: mps
Fixed test subjects: ['S004', 'S015', 'S023', 'S029', 'S031', 'S042', 'S055', 'S071', 'S082', 'S095']


# CELL 03 — Binary dataset loader
Uses the actual numeric T1/T2 event IDs returned by MNE, while mapping them to the binary machine-learning labels 0/1. This preserves the event-ID fix from the earlier notebook.

In [3]:
# CELL 04 — EEGMMIDB binary dataset
class EEGMMIDB_Dataset(Dataset):
    def __init__(self, data_dir, subjects, runs=RUNS, tmin=TMIN, tmax=TMAX):
        self.data_dir=str(data_dir); self.subjects=list(subjects); self.runs=list(runs)
        self.tmin=tmin; self.tmax=tmax
        self.epochs=[]; self.labels=[]; self.subject_ids=[]; self.run_ids=[]
        self.load_data()
    @staticmethod
    def _find_event_code(event_id_dict, target_name):
        target_name=str(target_name).strip().upper()
        for description, code in event_id_dict.items():
            desc=str(description).strip().upper()
            if desc==target_name: return int(code)
            if desc.startswith(target_name):
                remainder=desc[len(target_name):]
                if remainder=="" or remainder.startswith(("/","-","_")) or remainder.isspace(): return int(code)
        return None
    def load_data(self):
        total_loaded=0
        for sub in self.subjects:
            sub_folder=f"S{sub:03d}"; sub_path=os.path.join(self.data_dir, sub_folder)
            if not os.path.isdir(sub_path): continue
            for run in self.runs:
                edf_file=os.path.join(sub_path, f"{sub_folder}R{run:02d}.edf")
                if not os.path.isfile(edf_file): continue
                try:
                    raw=mne.io.read_raw_edf(edf_file, preload=True, verbose=False)
                    if len(raw.ch_names)<N_CHANNELS: continue
                    raw.pick(raw.ch_names[:N_CHANNELS]); raw.resample(FS, npad="auto")
                    events,event_id_dict=mne.events_from_annotations(raw, verbose=False)
                    t1_code=self._find_event_code(event_id_dict,"T1"); t2_code=self._find_event_code(event_id_dict,"T2")
                    if t1_code is None or t2_code is None:
                        print(f"[WARN] {sub_folder} R{run:02d}: T1/T2 not found. Available={list(event_id_dict.keys())}"); continue
                    ep=mne.Epochs(raw, events, event_id={"T1":t1_code,"T2":t2_code},
                                  tmin=self.tmin, tmax=self.tmax-1.0/FS, baseline=None, preload=True,
                                  reject_by_annotation=True, verbose=False)
                    if len(ep)==0: continue
                    data=ep.get_data(); actual_codes=ep.events[:,-1]
                    for i in range(len(data)):
                        if int(actual_codes[i])==t1_code: label=0
                        elif int(actual_codes[i])==t2_code: label=1
                        else: continue
                        self.epochs.append(data[i].astype(np.float32)); self.labels.append(label)
                        self.subject_ids.append(int(sub-1)); self.run_ids.append(int(run)); total_loaded+=1
                    print(f"[OK] {sub_folder} R{run:02d} | T1={t1_code}->Left | T2={t2_code}->Right | epochs={len(ep)}")
                except Exception as e:
                    print(f"[WARN] {sub_folder} R{run:02d}: {type(e).__name__}: {e}")
        print(f"\nDataset loading complete: {total_loaded} binary trials")
    def __len__(self): return len(self.epochs)
    def __getitem__(self, idx):
        x=torch.tensor(self.epochs[idx], dtype=torch.float32); y=torch.tensor(self.labels[idx], dtype=torch.long); s=torch.tensor(self.subject_ids[idx], dtype=torch.long)
        mean=x.mean(dim=1, keepdim=True); std=x.std(dim=1, keepdim=True); x=(x-mean)/(std+1e-6)
        return x,y,s

def discover_available_subjects(data_dir,max_subjects=TOTAL_SUBJECTS):
    return [s for s in range(1,max_subjects+1) if os.path.isdir(Path(data_dir)/f"S{s:03d}")]

In [4]:
# CELL 05 — Dataset sanity check
available=discover_available_subjects(DATA_DIR,TOTAL_SUBJECTS)
missing=[s for s in TEST_SUBJECTS if s not in available]
print("Available subjects:",len(available))
if missing: raise RuntimeError(f"Missing fixed test subjects: {missing}")
print("[OK] All fixed evaluation subjects are present.")

Available subjects: 109
[OK] All fixed evaluation subjects are present.


In [5]:
# CELL 06 — Gradient reversal
class GradientReversalLayer(torch.autograd.Function):
    @staticmethod
    def forward(ctx,x,lambda_grl): ctx.lambda_grl=lambda_grl; return x.view_as(x)
    @staticmethod
    def backward(ctx,grad_output): return grad_output.neg()*ctx.lambda_grl,None
def grl(x,lambda_grl=1.0): return GradientReversalLayer.apply(x,lambda_grl)

In [6]:
# CELL 07 — Original Sinc filter bank
class SincFilterBank(nn.Module):
    def __init__(self,in_channels=22,num_filters=10,kernel_size=81,sample_rate=250):
        super().__init__(); self.num_filters=num_filters; self.kernel_size=kernel_size; self.sample_rate=sample_rate
        self.f1=nn.Parameter(torch.rand(num_filters)*10+5); self.f2=nn.Parameter(torch.rand(num_filters)*20+15)
    def forward(self,x):
        B,C,T=x.shape; n=torch.arange(-(self.kernel_size//2),(self.kernel_size//2)+1,device=x.device,dtype=x.dtype); filters=[]
        for i in range(self.num_filters):
            lo=torch.minimum(self.f1[i],self.f2[i]-1e-3).clamp(0.5,70.0); hi=torch.maximum(self.f2[i],self.f1[i]+1e-3).clamp(1.0,95.0)
            f1_scaled=lo/self.sample_rate; f2_scaled=hi/self.sample_rate
            w=2*f2_scaled*torch.sinc(2*f2_scaled*n)-2*f1_scaled*torch.sinc(2*f1_scaled*n)
            filters.append(w.unsqueeze(0).unsqueeze(0))
        filters=torch.cat(filters,dim=0); out=F.conv1d(x.reshape(B*C,1,T),filters,padding="same")
        return out.reshape(B,C,self.num_filters,T).permute(0,2,1,3)

In [7]:
# CELL 08 — Original DGNN
class DGNN(nn.Module):
    def __init__(self,num_filters=10,in_nodes=22,out_nodes=64):
        super().__init__(); self.W_Q=nn.Linear(num_filters,num_filters); self.W_K=nn.Linear(num_filters,num_filters); self.W_V=nn.Linear(in_nodes,out_nodes); self.num_filters=num_filters
    def forward(self,x):
        B,num_bands,C,T=x.shape; x_flat=x.mean(dim=-1).transpose(1,2); Q=self.W_Q(x_flat); K=self.W_K(x_flat)
        A=torch.matmul(Q,K.transpose(-2,-1))/math.sqrt(self.num_filters); A=F.softmax(A,dim=-1)
        I=torch.eye(C,device=x.device,dtype=x.dtype).unsqueeze(0); A_hat=A+I; degree=A_hat.sum(dim=-1).clamp_min(1e-6)
        D_hat_inv_sqrt=torch.diag_embed(1.0/torch.sqrt(degree)); norm_A=D_hat_inv_sqrt@A_hat@D_hat_inv_sqrt
        x_trans=x.permute(0,1,3,2); out=torch.einsum("bij,bntj->bnti",norm_A,x_trans); out=F.elu(self.W_V(out))
        return out.permute(0,1,3,2)

In [8]:
# CELL 09 — Original BiGRU + SE attention
class SimplifiedBiMamba(nn.Module):
    def __init__(self,d_model=64):
        super().__init__(); self.ssm=nn.GRU(input_size=d_model,hidden_size=d_model//2,batch_first=True,bidirectional=True)
    def forward(self,x):
        out,_=self.ssm(x.transpose(1,2)); return out.transpose(1,2)

class SEAttention(nn.Module):
    def __init__(self,channel=64,reduction=16):
        super().__init__(); self.fc=nn.Sequential(nn.Linear(channel,channel//reduction,bias=False),nn.ReLU(inplace=True),nn.Linear(channel//reduction,channel,bias=False),nn.Sigmoid())
    def forward(self,x):
        b,c,_=x.size(); y=x.mean(dim=2); s=self.fc(y).view(b,c,1); return (x*s.expand_as(x)).mean(dim=2)

In [9]:
# CELL 10 — Original model, binary classifier only
class S3MambaDA(nn.Module):
    def __init__(self,num_classes=2,num_subjects=109):
        super().__init__()
        self.sinc_filter=SincFilterBank(in_channels=22,num_filters=10,kernel_size=81,sample_rate=250)
        self.dgnn=DGNN(num_filters=10,in_nodes=22,out_nodes=64)
        self.mamba=SimplifiedBiMamba(d_model=64)
        self.se_attention=SEAttention(channel=64)
        self.classifier=nn.Sequential(nn.BatchNorm1d(64),nn.Linear(64,num_classes))
        self.domain_classifier=nn.Sequential(nn.Linear(64,32),nn.ReLU(),nn.Linear(32,num_subjects))
        self.supcon_proj=nn.Sequential(nn.Linear(64,128),nn.ReLU(),nn.Linear(128,128))
    def forward(self,x,lambda_grl=1.0):
        f_out=self.sinc_filter(x); s_out=self.dgnn(f_out); pool_out=s_out.mean(dim=1); t_out=self.mamba(pool_out); z=self.se_attention(t_out)
        class_logits=self.classifier(z); domain_logits=self.domain_classifier(grl(z,lambda_grl)); z_proj=F.normalize(self.supcon_proj(z),p=2,dim=1)
        return class_logits,domain_logits,z_proj

In [10]:
# CELL 11 — SupCon loss + AdaBN
class SupConLoss(nn.Module):
    def __init__(self,temperature=0.07): super().__init__(); self.temperature=temperature
    def forward(self,features,labels):
        device=features.device; sim=torch.matmul(features,features.T)/self.temperature; labels=labels.view(-1,1); mask=torch.eq(labels,labels.T).float().to(device); logits_mask=torch.ones_like(mask); logits_mask.fill_diagonal_(0); mask=mask*logits_mask
        exp_logits=torch.exp(sim)*logits_mask; log_prob=sim-torch.log(exp_logits.sum(1,keepdim=True)+1e-6); positives=mask.sum(1); mean_log_prob_pos=(mask*log_prob).sum(1)/(positives+1e-6); valid=positives>0
        return -mean_log_prob_pos[valid].mean() if valid.any() else torch.zeros((),device=device,requires_grad=True)

def apply_adabn(model,target_dataloader,device,adaptation_trials=20):
    model.eval(); target_samples=[]; trials_count=0
    for x,_,_ in target_dataloader:
        target_samples.append(x); trials_count+=x.size(0)
        if trials_count>=adaptation_trials: break
    if not target_samples: return model
    target_x=torch.cat(target_samples,dim=0)[:adaptation_trials].to(device)
    bn_modules=[m for m in model.modules() if isinstance(m,nn.modules.batchnorm._BatchNorm)]
    if not bn_modules: return model
    for module in bn_modules:
        module.reset_running_stats(); module.momentum=1.0; module.train()
    with torch.no_grad(): _=model(target_x,lambda_grl=0.0)
    for module in bn_modules: module.momentum=0.1; module.eval()
    model.eval(); return model

In [11]:
# CELL 12 — Smoke test
model=S3MambaDA(num_classes=2,num_subjects=DOMAIN_CLASSES).to(DEVICE)
x=torch.randn(2,N_CHANNELS,int(FS*TMAX),device=DEVICE)
with torch.no_grad(): class_logits,domain_logits,z_proj=model(x,lambda_grl=0.0)
print("Input:",tuple(x.shape)); print("Class logits:",tuple(class_logits.shape)); print("Domain logits:",tuple(domain_logits.shape)); print("Projection:",tuple(z_proj.shape)); print("Params:",sum(p.numel() for p in model.parameters()))
assert class_logits.shape==(2,2) and domain_logits.shape==(2,109) and z_proj.shape==(2,128)
print("[OK] Smoke test passed.")
del model,x,class_logits,domain_logits,z_proj

Input: (2, 22, 1000)
Class logits: (2, 2)
Domain logits: (2, 109)
Projection: (2, 128)
Params: 51807
[OK] Smoke test passed.


# CELL 13 — Training and evaluation
The next cell preserves the original 100-epoch objective: classification CE + 1.0×domain CE + 0.5×SupCon, original cosine scheduler, GRL schedule, and target AdaBN. The only task/output change is binary classification.

In [12]:
# CELL 14 — Fixed 10-subject binary reproduction
def run_reproduction(data_dir=DATA_DIR,seed=SEED):
    seed_everything(seed); available=discover_available_subjects(data_dir,TOTAL_SUBJECTS); rng=random.Random(seed)
    fold_rows=[]; pred_rows=[]; epoch_rows=[]; embedding_rows=[]
    for fold_idx,test_subject in enumerate(TEST_SUBJECTS,start=1):
        print("="*80); print(f"FOLD {fold_idx}/{len(TEST_SUBJECTS)} | TEST SUBJECT S{test_subject:03d}"); print("="*80)
        remaining=[s for s in available if s!=test_subject]; train_subjects=rng.sample(remaining,min(NUM_TRAIN_SUBJECTS,len(remaining)))
        train_dataset=EEGMMIDB_Dataset(data_dir,train_subjects); test_dataset=EEGMMIDB_Dataset(data_dir,[test_subject])
        if len(train_dataset)==0 or len(test_dataset)==0: print("[WARN] Empty fold; skipping."); continue
        train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=False)
        test_loader=DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=False)
        model=S3MambaDA(num_classes=2,num_subjects=TOTAL_SUBJECTS).to(DEVICE)
        criterion_cls=nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING); criterion_domain=nn.CrossEntropyLoss(); criterion_supcon=SupConLoss(SUPCON_TEMP)
        optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY); scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=NUM_EPOCHS,eta_min=1e-5)
        use_amp=DEVICE.type=="cuda"; scaler=torch.amp.GradScaler("cuda",enabled=use_amp)
        for epoch in range(NUM_EPOCHS):
            model.train(); epoch_loss=epoch_cls=epoch_domain=epoch_supcon=0.0; seen=0; total_batches=len(train_loader)
            for batch_idx,(x,y,s) in enumerate(train_loader):
                x=x.to(DEVICE); y=y.to(DEVICE); s=s.to(DEVICE); p=float(batch_idx+epoch*total_batches)/max(1,NUM_EPOCHS*total_batches); lam=2.0/(1.0+np.exp(-10.0*p))-1.0; optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type,enabled=use_amp):
                    cl,dl,zp=model(x,lambda_grl=lam); lcls=criterion_cls(cl,y); ldom=criterion_domain(dl,s); lsup=criterion_supcon(zp,y); ltotal=lcls+DOMAIN_WEIGHT*ldom+SUPCON_WEIGHT*lsup
                scaler.scale(ltotal).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP); scaler.step(optimizer); scaler.update()
                bs=x.size(0); seen+=bs; epoch_loss+=float(ltotal.detach().cpu())*bs; epoch_cls+=float(lcls.detach().cpu())*bs; epoch_domain+=float(ldom.detach().cpu())*bs; epoch_supcon+=float(lsup.detach().cpu())*bs
            scheduler.step(); epoch_rows.append({"fold":fold_idx,"test_subject":test_subject,"epoch":epoch+1,"loss_total":epoch_loss/max(1,seen),"loss_cls":epoch_cls/max(1,seen),"loss_domain":epoch_domain/max(1,seen),"loss_supcon":epoch_supcon/max(1,seen),"lr":optimizer.param_groups[0]["lr"]})
            if epoch==0 or (epoch+1)%10==0: print(f"Epoch {epoch+1:03d}/{NUM_EPOCHS} | Loss {epoch_loss/max(1,seen):.4f}")
        model=apply_adabn(model,test_loader,DEVICE,adaptation_trials=len(test_dataset)); model.eval(); preds=[]; labels=[]; probs=[]; embeds=[]
        with torch.no_grad():
            for x,y,s in test_loader:
                x=x.to(DEVICE); cl,_,zp=model(x,lambda_grl=0.0); pr=torch.softmax(cl,dim=1); pd=torch.argmax(pr,dim=1); preds.extend(pd.cpu().numpy().tolist()); labels.extend(y.numpy().tolist()); probs.append(pr.cpu().numpy()); embeds.append(zp.cpu().numpy())
        probs=np.concatenate(probs,axis=0); embeds=np.concatenate(embeds,axis=0); acc=accuracy_score(labels,preds); bacc=balanced_accuracy_score(labels,preds); kap=cohen_kappa_score(labels,preds); ra=roc_auc_score(labels,probs[:,1]) if len(np.unique(labels))==2 else np.nan; rep=classification_report(labels,preds,labels=[0,1],target_names=CLASS_NAMES,output_dict=True,zero_division=0)
        fold_rows.append({"fold":fold_idx,"test_subject":test_subject,"n_train_trials":len(train_dataset),"n_test_trials":len(test_dataset),"accuracy":acc,"balanced_accuracy":bacc,"kappa":kap,"roc_auc":ra,"precision_macro":rep["macro avg"]["precision"],"recall_macro":rep["macro avg"]["recall"],"f1_macro":rep["macro avg"]["f1-score"]})
        for i in range(len(labels)):
            pred_rows.append({"fold":fold_idx,"test_subject":test_subject,"true_label":int(labels[i]),"pred_label":int(preds[i]),"prob_left":float(probs[i,0]),"prob_right":float(probs[i,1])}); embedding_rows.append({"fold":fold_idx,"test_subject":test_subject,"true_label":int(labels[i]),**{f"z_{j}":float(embeds[i,j]) for j in range(embeds.shape[1])}})
        print(f"Subject S{test_subject:03d} | Accuracy={acc*100:.2f}% | BalancedAcc={bacc*100:.2f}% | Kappa={kap:.4f} | ROC-AUC={ra:.4f}")
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    fold_df=pd.DataFrame(fold_rows); pred_df=pd.DataFrame(pred_rows); epoch_df=pd.DataFrame(epoch_rows); emb_df=pd.DataFrame(embedding_rows)
    fold_df.to_csv(RESULTS_DIR/"fold_metrics.csv",index=False); pred_df.to_csv(RESULTS_DIR/"test_predictions.csv",index=False); epoch_df.to_csv(RESULTS_DIR/"epoch_history.csv",index=False); emb_df.to_csv(RESULTS_DIR/"test_embeddings.csv",index=False)
    if len(fold_df):
        summary={"mean_accuracy":float(fold_df.accuracy.mean()),"population_sd_accuracy":float(fold_df.accuracy.std(ddof=0)),"mean_kappa":float(fold_df.kappa.mean()),"mean_roc_auc":float(fold_df.roc_auc.mean()),"total_test_samples":int(fold_df.n_test_trials.sum()),"num_folds_completed":int(len(fold_df)),"test_subjects":TEST_SUBJECTS,"classes":CLASS_NAMES}
        summary["pooled_accuracy"]=float(accuracy_score(pred_df.true_label,pred_df.pred_label))
    else: summary={"error":"No completed folds"}
    with open(RESULTS_DIR/"summary.json","w") as f: json.dump(summary,f,indent=2)
    print("\nFINAL BINARY REPRODUCTION SUMMARY"); print(json.dumps(summary,indent=2)); return fold_df,pred_df,epoch_df,emb_df

In [ ]:
# CELL 15 — Run
fold_df,pred_df,epoch_df,emb_df=run_reproduction()

FOLD 1/10 | TEST SUBJECT S004
[OK] S083 R04 | T1=2->Left | T2=3->Right | epochs=15
[OK] S083 R08 | T1=2->Left | T2=3->Right | epochs=15
[OK] S083 R12 | T1=2->Left | T2=3->Right | epochs=15
[OK] S016 R04 | T1=2->Left | T2=3->Right | epochs=15
[OK] S016 R08 | T1=2->Left | T2=3->Right | epochs=15
[OK] S016 R12 | T1=2->Left | T2=3->Right | epochs=15
[OK] S005 R04 | T1=2->Left | T2=3->Right | epochs=15
[OK] S005 R08 | T1=2->Left | T2=3->Right | epochs=15
[OK] S005 R12 | T1=2->Left | T2=3->Right | epochs=15
[OK] S096 R04 | T1=2->Left | T2=3->Right | epochs=15
[OK] S096 R08 | T1=2->Left | T2=3->Right | epochs=15
[OK] S096 R12 | T1=2->Left | T2=3->Right | epochs=15
[OK] S037 R04 | T1=2->Left | T2=3->Right | epochs=15
[OK] S037 R08 | T1=2->Left | T2=3->Right | epochs=15
[OK] S037 R12 | T1=2->Left | T2=3->Right | epochs=15
[OK] S033 R04 | T1=2->Left | T2=3->Right | epochs=15
[OK] S033 R08 | T1=2->Left | T2=3->Right | epochs=15
[OK] S033 R12 | T1=2->Left | T2=3->Right | epochs=15
[OK] S030 R04 | 

In [ ]:
# CELL 16 — Results table
if not fold_df.empty:
    out=fold_df.copy(); out["accuracy"]*=100; out["balanced_accuracy"]*=100; out["roc_auc"]*=100; out["f1_macro"]*=100
    display(out[["test_subject","n_test_trials","accuracy","balanced_accuracy","kappa","roc_auc","f1_macro"]])
    print(f"Mean accuracy: {fold_df.accuracy.mean()*100:.2f}%")
    print(f"Population SD: {fold_df.accuracy.std(ddof=0)*100:.2f} pp")
    print(f"Mean Kappa: {fold_df.kappa.mean():.4f}")
    print(f"Pooled accuracy: {accuracy_score(pred_df.true_label,pred_df.pred_label)*100:.2f}%")

In [ ]:
# CELL 17 — Binary confusion matrix + ROC
if not pred_df.empty:
    cm=confusion_matrix(pred_df.true_label,pred_df.pred_label,labels=[0,1]); cmn=cm/np.maximum(cm.sum(axis=1,keepdims=True),1)
    fig,ax=plt.subplots(figsize=(6,5)); im=ax.imshow(cmn,interpolation="nearest"); ax.set_title("Binary Normalized Confusion Matrix"); ax.set_xlabel("Predicted class"); ax.set_ylabel("True class"); ax.set_xticks([0,1],CLASS_NAMES,rotation=20,ha="right"); ax.set_yticks([0,1],CLASS_NAMES)
    for i in range(2):
        for j in range(2): ax.text(j,i,f"{cmn[i,j]*100:.1f}%",ha="center",va="center")
    fig.colorbar(im,ax=ax); fig.tight_layout(); fig.savefig(FIG_DIR/"Binary_ConfusionMatrix.png",dpi=400,bbox_inches="tight"); plt.show()
    if pred_df.true_label.nunique()==2:
        fpr,tpr,_=roc_curve(pred_df.true_label,pred_df.prob_right); ra=auc(fpr,tpr); fig,ax=plt.subplots(figsize=(6,5)); ax.plot(fpr,tpr,linewidth=2,label=f"Right Hand MI (AUC={ra:.3f})"); ax.plot([0,1],[0,1],linestyle="--",linewidth=1); ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate"); ax.set_title("Binary ROC Curve"); ax.legend(); ax.grid(True,alpha=0.25); fig.tight_layout(); fig.savefig(FIG_DIR/"Binary_ROC_AUC.png",dpi=400,bbox_inches="tight"); plt.show()